# GRM shard batch submission (Google Batch / dsub)

Moves GRM shard construction off the interactive VM and onto Google Batch,
via `dsub` -- one task per shard, each on its own machine. Three stages,
each meant to be run in order, and each a checkpoint before spending more:

1. **Troubleshooting job** -- the smallest possible `dsub` submission
   (no plink, no real inputs), just to confirm submission itself works
   (auth, project/region config, logging bucket permissions, the
   `google-batch` provider) before anything plink-specific can go wrong
   and get confused with dsub-specific problems.
2. **Validation batch** -- a handful of real shards (5, spread across a
   candidate `N_SHARDS`) with the real plink command, to get real
   per-task timing/cost before committing to the full run.
3. **Full run** -- not built yet. Comes after (2) confirms timing/cost are
   sane, and after the fixed-vs-variable-cost question from
   `grm_shard_timing.ipynb` is resolved (does per-shard time stay roughly
   flat going from 2000 shards down to ~300? if so, fewer/larger shards is
   strictly better -- see that notebook's "Next steps").

**Everything here is a first draft.** `dsub` provider names and exact
machine-type string formatting are version-sensitive -- expect to debug
the troubleshooting job itself before it succeeds.

## Prerequisites

`dsub` submits to Google Batch from wherever this cell runs (this
notebook's VM), but the actual work happens on separate Batch-provisioned
machines -- neither the interactive VM's compute resource nor its plink
install matter for what follows. Needs `dsub` installed and `gcloud`
already authenticated to the right project (both should already be true on
a Verily Workbench VM, but confirmed explicitly below rather than assumed).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Fill in `PROJECT_ID`/`REGION` from the `gcloud config` output above if they
didn't print automatically. `MEMORY_MB` is a placeholder -- replace with
the real `max_rss_gb` measured in `grm_shard_timing.ipynb` (with
`--read-freq`/`--memory` applied) plus ~20-30% headroom before running the
validation batch for real; the troubleshooting job below doesn't use it.

In [ ]:
import os

PROJECT_ID = "TODO-your-project-id"
REGION = "us-central1"   # TODO -- confirm against gcloud config output above

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
# dsub needs gs:// URIs, not the gcsfuse-mounted local path -- TODO fill in the
# actual gs:// bucket resource name (visible via `gcloud config` above, or
# `env | grep WORKSPACE`/`wb resource list` per the root README)
WORKSPACE_BUCKET_GS = "gs://TODO-your-bucket-resource-name/shared-env-pilot"

BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/data/03_grm_shards"

GRM_INPUT_DIR = f"{BUCKET_DIR}/grm_input"
GRM_INPUT_DIR_GS = f"{BUCKET_DIR_GS}/grm_input"

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = f"{BUCKET_DIR}/{BED_NAME}"

MEMORY_MB = 73728   # placeholder -- replace with measured max_rss_gb (grm_shard_timing.ipynb) + headroom
MACHINE_VCPUS = 2    # plink's --make-grm-bin is single-threaded -- more vCPUs buys nothing

print(BUCKET_DIR_GS)

## Stage a clean input folder (one-time)

Each `dsub` task downloads whatever's in this folder -- keep it to just the
4 files plink actually needs, not the whole `03_grm_shards/` bucket
contents (which also has the per-chromosome intermediates, logs, etc.).

In [ ]:
%%bash -s "$BED_PREFIX" "$GRM_INPUT_DIR"
set -e
BED_PREFIX=$1
GRM_INPUT_DIR=$2

mkdir -p "$GRM_INPUT_DIR"
cp "${BED_PREFIX}".bed "${BED_PREFIX}".bim "${BED_PREFIX}".fam "${BED_PREFIX}_freq.frq" "$GRM_INPUT_DIR/"
ls -lh "$GRM_INPUT_DIR"

## Stage 1 -- troubleshooting job

The smallest possible submission: no real inputs, no plink, just
`echo`/`hostname`/`date` on the cheapest default machine. If this doesn't
come back `SUCCESS` via `dstat` below, the problem is in dsub/Batch/auth
setup, not in anything plink- or GRM-specific -- worth fully resolving
before moving to Stage 2.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --name "grm-troubleshoot" \
  --command 'echo "hello from dsub"; hostname; date' \
  > /tmp/troubleshoot_job_id.txt

cat /tmp/troubleshoot_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID"
set -e
PROJECT_ID=$1
JOB_ID=$(tail -1 /tmp/troubleshoot_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --jobs "$JOB_ID" --status '*' --full

## Stage 2 -- validation batch (5 real shards)

Only run this once Stage 1 comes back `SUCCESS`. Candidate production
shard count is `N_SHARDS = 300` (per the "fewer, larger shards" direction
from `grm_shard_timing.ipynb` -- adjust once that notebook's
fixed-vs-variable-cost test is back). `VALIDATION_KS` samples 5 shards
spread across the range, same reasoning as that notebook's `SAMPLE_KS`,
rather than committing to all 300 before knowing real per-task cost/timing
on Batch specifically (staging + provisioning overhead per task is not
something the interactive-VM timing captured).

In [ ]:
N_SHARDS = 300   # candidate production count -- adjust per grm_shard_timing.ipynb's fixed-vs-variable-cost test
VALIDATION_KS = [1, 75, 150, 225, 300]   # spread across the full range, not clustered at one end

TASKS_PATH = "/tmp/grm_validation_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env K\t--env N_SHARDS\n")
    for k in VALIDATION_KS:
        f.write(f"{k}\t{N_SHARDS}\n")

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS" "$GRM_INPUT_DIR_GS" "$MACHINE_VCPUS" "$MEMORY_MB"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
GRM_INPUT_DIR_GS=$4
MACHINE_VCPUS=$5
MEMORY_MB=$6

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --name "grm-shard-validation" \
  --machine-type "n1-custom-${MACHINE_VCPUS}-${MEMORY_MB}" \
  --disk-size 100 \
  --input-recursive BED_DIR="$GRM_INPUT_DIR_GS" \
  --output-recursive OUT_DIR="${BUCKET_DIR_GS}/shards/shard_\${K}_of_\${N_SHARDS}" \
  --command '
    set -e
    BIN_DIR=/tmp/bin
    mkdir -p "$BIN_DIR"
    PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
    wget -q -O /tmp/plink.zip "$PLINK_URL"
    unzip -o -q /tmp/plink.zip plink -d "$BIN_DIR"
    chmod +x "$BIN_DIR/plink"

    BED_PREFIX="${BED_DIR}/genome_wide_round2b_thinned_bed"
    FREQ_PATH="${BED_DIR}/genome_wide_round2b_thinned_bed_freq.frq"

    "$BIN_DIR/plink" \
      --bfile "$BED_PREFIX" \
      --make-grm-bin \
      --parallel "${K}" "${N_SHARDS}" \
      --read-freq "$FREQ_PATH" \
      --memory '"$MEMORY_MB"' \
      --out "${OUT_DIR}/grm_shard_${K}_of_${N_SHARDS}"
  ' \
  --tasks /tmp/grm_validation_tasks.tsv \
  > /tmp/validation_job_id.txt

cat /tmp/validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID"
set -e
PROJECT_ID=$1
JOB_ID=$(tail -1 /tmp/validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --jobs "$JOB_ID" --status '*' --full

## Next steps (tomorrow)

1. Fill in `PROJECT_ID`/`REGION`/`WORKSPACE_BUCKET_GS` in the Inputs cell.
2. Run Stage 1 -- confirm `dstat` shows `SUCCESS` before touching anything
   plink-related. If it fails, the error is in dsub/Batch/auth setup
   (wrong provider name, missing permissions, project/region mismatch),
   not GRM-specific.
3. Fill in the real `MEMORY_MB` (measured `max_rss_gb` from
   `grm_shard_timing.ipynb`'s driver, plus headroom), then run Stage 2.
4. Compare Stage 2's real per-task wall-clock/cost against the interactive
   `grm_shard_timing.ipynb` numbers -- Batch tasks have extra overhead
   (VM provisioning, downloading plink + inputs fresh each time) the
   interactive timing didn't capture, worth knowing before extrapolating
   cost for the full run.
5. Once (2)-(4) look sane and `N_SHARDS` is settled, scale `VALIDATION_KS`
   up to the full `range(1, N_SHARDS + 1)` for the real run -- not built
   as its own cell yet, deliberately, until the above is confirmed.